In [8]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [9]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1772642448394, experiment_id='2', last_update_time=1772642448394, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1772533161579, experiment_id='1', last_update_time=1772533161579, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1772443737985, experiment_id='0', last_update_time=1772443737985, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
client.create_experiment(name="my-cool-experiment")

MlflowException: Experiment(name=my-cool-experiment) already exists. Error: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: experiments.name
[SQL: INSERT INTO experiments (name, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?)]
[parameters: ('my-cool-experiment', None, 'active', 1772660240760, 1772660240760)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [10]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids="1",
    filter_string="metrics.rmse < 6.5",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [11]:
for run in runs:
    print(f"model : {run.info.run_id}, rmse : {run.data.metrics['rmse']:.4f}")

model : af103ba7dcaf45fa9abb6bd0fefb33a7, rmse : 6.3268
model : a86e35c3d1a34389a402906ea805bbbe, rmse : 6.3543
model : 0952f87dadbc450ab39bbc679e1dee54, rmse : 6.3543
model : 130158125cb64f0ca90e55b6421a7673, rmse : 6.3543
model : d88d264068a24ab3b7e7d7705a1bbe8a, rmse : 6.4729


In [12]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [14]:
run_id = "a86e35c3d1a34389a402906ea805bbbe"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/03/04 22:02:25 WARNING mlflow.tracking._model_registry.fluent: Run with id a86e35c3d1a34389a402906ea805bbbe has no artifacts at artifact path 'model', registering model based on models:/m-b510586263c14f8482e5e1022e1e4537 instead


Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1772661745920, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1772661745920, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='a86e35c3d1a34389a402906ea805bbbe', run_link=None, source='models:/m-b510586263c14f8482e5e1022e1e4537', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [15]:
model_uri

'runs:/a86e35c3d1a34389a402906ea805bbbe/model'

In [ ]:
#client.search_registered_models()
model_name = "nyc-taxi-regressor"

""" obsolète
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")
"""

all_versions = client.search_model_versions(f"name='{model_name}'")

for version in all_versions:
    print(f"version: {version.version}, aliases: {version.aliases}, run_id: {version.run_id}")

version: 3, stage: None
version: 3, aliases: [], run_id: a86e35c3d1a34389a402906ea805bbbe
version: 2, aliases: [], run_id: 
version: 1, aliases: [], run_id: 


/tmp/ipykernel_2749/1725489859.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [ ]:
"""
#obsolète
client.transition_model_version_stage(
    name = model_name,
    version=3,
    stage="Staging",
    archive_existing_versions=False
)
"""
model_version = 3
# il faudrait mettre alias plutot que stage du coup
new_stage = "Staging"
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=1
)

In [30]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=['staging'], creation_timestamp=1772661745920, current_stage='Staging', deployment_job_state=None, description='The model version 3 was transitioned to Staging on 2026-03-04', last_updated_timestamp=1772663631554, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='a86e35c3d1a34389a402906ea805bbbe', run_link=None, source='models:/m-b510586263c14f8482e5e1022e1e4537', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [51]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)

"""obsolete
def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}
"""

def test_model(name, alias_or_version, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}@{alias_or_version}")  # alias ou version
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}


In [44]:
df = read_dataframe("./data/green_tripdata_2021-03.parquet")

In [45]:
run_id = "0952f87dadbc450ab39bbc679e1dee54"
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [46]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [47]:
X_test = preprocess(df, dv)

In [48]:
target = "duration"
y_test = df[target].values

In [54]:
%time test_model(name=model_name, alias_or_version="champion", X_test=X_test, y_test=y_test)

CPU times: user 7.52 s, sys: 197 ms, total: 7.71 s
Wall time: 9.17 s


{'rmse': 6.3087433820101975}

In [55]:
%time test_model(name=model_name, alias_or_version="staging", X_test=X_test, y_test=y_test)

CPU times: user 7.14 s, sys: 10.9 ms, total: 7.16 s
Wall time: 4.54 s


{'rmse': 6.3087433820101975}

In [ ]:
# comme le staging est plus rapide on devrait le passer en champion
# retrouver la version numéro X qui correspond au staging
"""
best_version = X 
new_alias = "champion"
client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=best_version
)
"""